# Train a seven-day tram forecast

One training sample is a **route + observation cutoff**. The encoder reads the
preceding 21 days once, and the prediction head uses that context for each requested
future day. Start with seven days and the boarding-only model; select `full` to
include raw events. Both use the same fitting split and weekly-profile comparison.

Run cells in order. The **Train** cell performs real optimization when you execute
it. Preparation needs the local raw CSVs and labels listed in the README. No data,
weights, or executed outputs are embedded in this notebook.

Install the package with the `notebook` extra, select that environment as your
Jupyter kernel, and restart the kernel after changing PyTorch. See the repository
README for installation commands.

In [ ]:
import json
import os
import sys
from dataclasses import replace
from pathlib import Path

# Works when Jupyter starts at the repository root or inside notebooks/.
ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / 'src/tram_forecast').is_dir() and (path / 'pyproject.toml').is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError('Start Jupyter inside the tram forecasting repository.')
os.chdir(ROOT)

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import display

from tram_forecast.checkpoint import load_model
from tram_forecast.dataset import ForecastDataset
from tram_forecast.evaluate import evaluate_model, fixed_forecast, metric_report, weekly_profile
from tram_forecast.io import write_json
from tram_forecast.preprocess import prepare
from tram_forecast.settings import Settings
from tram_forecast.smoke import smoke
from tram_forecast.storage import Store
from tram_forecast.train import train

print('Repository:', ROOT)
print('Kernel Python:', sys.executable)
print('PyTorch:', torch.__version__, '| Wheel CUDA runtime:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Configure the run

`BATCH_SIZE` counts contexts. Each context has up to `FORECAST_DAYS` targets;
near the fitting boundary the group is shorter. Gradients are weighted by the
number of requested days. `ACCUMULATION=8` combines eight context microbatches.

Change `RUN_NAME` for each experiment. To resume, set `RESUME` to that run's
`latest.pt`; the original run directory is used automatically. Keep the original
settings except the epoch limit. Changing horizon requires a fresh run, but can
reuse the prepared artifact. Dates remain fixed: fit January–August and evaluate
September 1–7 by default, using only August 11–31 as validation history.

In [ ]:
MODEL_KIND = 'boarding_only'  # 'full' adds raw-event processing
FORECAST_DAYS = 7            # Increase later, up to 61
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
EPOCHS = 30
BATCH_SIZE = 1
ACCUMULATION = 8
RUN_NAME = f'{MODEL_KIND}_{FORECAST_DAYS}days_v1'
RESUME = None                # Example: 'outputs/notebook_runs/boarding_only_7days_v1/latest.pt'

base = Settings.load(ROOT / 'configs/default.json')
settings = replace(
    base,
    training=replace(
        base.training,
        forecast_days=FORECAST_DAYS,
        device=DEVICE,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        accumulation=ACCUMULATION,
    ),
)
if MODEL_KIND not in ('boarding_only', 'full'):
    raise ValueError('MODEL_KIND must be boarding_only or full.')
if DEVICE.startswith('cuda') and not torch.cuda.is_available():
    raise RuntimeError('CUDA requested but unavailable; check the wheel, driver and kernel.')

# Edit settings.data paths above if your dataset lives outside this repository.
ARTIFACT = Path(settings.data.output_root) / 'validation'
RUN_DIR = Path(RESUME).resolve().parent if RESUME else ROOT / 'outputs/notebook_runs' / RUN_NAME
print('Model:', MODEL_KIND, '| Horizon:', FORECAST_DAYS, '| Device:', DEVICE)
print('Artifact:', ARTIFACT, '| Run:', RUN_DIR)
display(settings.to_dict())

## Prepare or reuse the fitting artifact

Preparation validates CSVs, fits vocabularies/scaling only on January–August,
and sorts raw events on disk. It may take time on the full dataset. Identical
inputs reuse an existing artifact; changed inputs require a new artifact path.
The boarding-only model skips event loading during training, although the shared
preparation command still processes raw sources.

In [ ]:
missing = [path for path in (*settings.data.raw_paths, *settings.data.label_paths)
           if not Path(path).is_file()]
if missing:
    raise FileNotFoundError('Missing dataset files: ' + ', '.join(missing))
ARTIFACT = prepare(settings, 'validation', ARTIFACT)
dataset = ForecastDataset(ARTIFACT, MODEL_KIND, settings.training.forecast_days)
store = dataset.store
print('Training contexts:', len(dataset))
print('First context:', dataset[0]['identity'])
print('First target group:', dataset[0]['target'].shape)
print('Raw/label mismatched hours:', store.metadata['raw_label_mismatched_hours'])

## Weekly-profile baseline and resource check

The baseline averages the three matching weekday/hour observations in the fixed
21-day history. It predicts the same horizon as the neural model. WAPE is lower
when forecasts are better. The smoke check runs forward/backward for typical and
busiest contexts, without an optimizer step. Reduce batch or event chunk budgets
if memory is insufficient; keep all events.

In [ ]:
from datetime import date

available_days = (date.fromisoformat(store.metadata['evaluation_end']) - store.end).days
if settings.training.forecast_days > available_days:
    raise ValueError('Requested horizon exceeds the available validation period.')
actual = np.asarray(store.evaluation_targets()).reshape(len(store.routes) + 1, available_days, 24)
actual = actual[:, :settings.training.forecast_days]
baseline_prediction = weekly_profile(store, settings.training.forecast_days)
baseline_report = metric_report(actual[:-1], baseline_prediction, store.routes)
display(baseline_report['global'])

In [ ]:
resource_report = smoke(settings, ARTIFACT, MODEL_KIND)
display(resource_report)

## Train

This cell runs training and prints one summary per epoch. Checkpoints, resolved
configuration and `history.json` are saved in `RUN_DIR`. Validation uses the same
fixed context at every epoch. The lowest validation WAPE selects `best.pt`.

To continue after interruption, set `RESUME` in the configuration cell and rerun
from there. Epoch-boundary resume reruns an interrupted partial epoch. A fresh
run refuses to overwrite existing checkpoints.

In [ ]:
best_checkpoint = train(
    settings, ARTIFACT, model_kind=MODEL_KIND, output=RUN_DIR, resume=RESUME,
)
print('Selected checkpoint:', best_checkpoint)

## Learning curves

In [ ]:
history = json.loads((RUN_DIR / 'history.json').read_text())
epochs = [row['epoch'] for row in history]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, [row['mae'] for row in history], marker='o')
axes[0].set(title='Training MAE', xlabel='Epoch', ylabel='Boardings')
axes[1].plot(epochs, [row['validation']['neural']['global']['wape'] for row in history], marker='o', label=MODEL_KIND)
if baseline_report['global']['wape'] is not None:
    axes[1].axhline(baseline_report['global']['wape'], linestyle='--', label='Weekly profile')
axes[1].set(title='Fixed-context validation WAPE', xlabel='Epoch', ylabel='WAPE')
axes[1].legend()
fig.tight_layout()
plt.show()

## Evaluate the best checkpoint

The horizon comes from the saved checkpoint. Metrics include per-route/per-lead
errors and route 5's separate zero fallback. Rerun the notebook with a new run name
and `MODEL_KIND='full'` to compare the raw-event model against boarding-only.

In [ ]:
model, payload = load_model(best_checkpoint, store, settings.training.device)
checkpoint_days = payload['settings']['training']['forecast_days']
report = evaluate_model(model, store, checkpoint_days)
write_json(RUN_DIR / 'evaluation.json', report)
display({
    'horizon_days': report['forecast_days'],
    'neural': report['neural']['global'],
    'weekly_profile': report['weekly_profile']['global'],
    'including_route5': report['full_grid_with_route5']['global'],
})

In [ ]:
ROUTE_TO_PLOT = store.routes[0]
route_index = store.routes.index(ROUTE_TO_PLOT)
prediction = fixed_forecast(model, store, checkpoint_days)
hours = np.arange(checkpoint_days * 24)
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(hours, actual[route_index, :checkpoint_days].reshape(-1), label='Observed')
ax.plot(hours, prediction[route_index].reshape(-1), label=MODEL_KIND)
ax.plot(hours, baseline_prediction[route_index, :checkpoint_days].reshape(-1), label='Weekly profile', alpha=0.7)
ax.set(title=f'Route {ROUTE_TO_PLOT}: {checkpoint_days}-day forecast from {store.end}',
       xlabel='Hours after fixed cutoff', ylabel='Boardings')
ax.legend()
fig.tight_layout()
plt.show()

## Extend the experiment

- Change `MODEL_KIND` and `RUN_NAME` to compare boarding-only and full models.
- Increase `FORECAST_DAYS` (for example to 14), choose a new run name, and retrain.
- Keep horizons and artifacts identical when comparing model variants; the CLI
  `compare` command checks this.
- Final January–October refitting and forecast export are separate steps after
  model selection; see the README. A seven-day checkpoint produces 1,680 rows,
  while the competition submission requires a model trained for all 61 days.

This notebook calls the package's training code so checkpointing, loss weighting,
and evaluation behavior stay consistent with the CLI.